# Drone navigation dataset: set up and collect the first trajectories

Genesis drone room from the Colab "Drone environment" notebook (same drone, SE(3) controller, FPV RGB-D camera).
For each trajectory: pick a start and a goal in the room, plan a collision-free path (A* on the floor plan),
fly the drone along it with a path follower, and record RGB, depth, state (proprio) and actions every 0.1 s.

**Where to run it on Savio:** Open OnDemand (ood.brc.berkeley.edu), Jupyter Server, *compute via Slurm*,
account `fc_auto`, partition `savio3_gpu`, QoS `gtx2080_gpu3_normal`, GRES `gpu:GTX2080TI:1`, 2 CPUs, 1 to 2 hours.
Pick the kernel **drone_env (Genesis)** (created by `setup/drone_setup.sh`).
When done: **File > Shut Down**, then delete the session under *My Interactive Sessions*. Closing the tab does not stop the charge.

Sections 1 to 3 need no GPU and no Genesis, so you can do them on the free shared Jupyter node first.

Code: `drone_sim/` (room, planner, follower, env, collect), loader `datasets/drone_dset.py`. Batch version: `scripts/drone/collect.py`, `slurm/drone_collect.sbatch`.

In [ ]:
#@title 0. Paths. GPU rendering must be set up before Genesis is imported.
import os, sys, json, time
REPO = os.path.expanduser("~/dino_wm")                     # your clone of IvanFenster/dino_wm (branch drone-env)
WM = os.environ.get("WM", f"/global/scratch/users/{os.environ['USER']}/arpl_wm")
OUT = f"{WM}/data/drone_raw_notebook"                     # raw episodes from this notebook
DSET = f"{WM}/data/drone_nav_notebook"                    # packed DINO-WM dataset
os.environ.setdefault("TRANSPORT_WORLD", f"{WM}/src/transport-world")
os.environ.setdefault("ARPL_EGL_DIR", f"{WM}/cache/egl_notebook")
sys.path.insert(0, REPO)

from drone_sim.egl import setup_gpu_rendering
setup_gpu_rendering()          # prints which libEGL_nvidia it registered, or warns about CPU rendering

import numpy as np
import matplotlib.pyplot as plt
from drone_sim.room import Room, RoomConfig, Box
from drone_sim.planner import PlannerConfig, plan_path, sample_start_goal
from drone_sim.follower import FollowerConfig
from drone_sim.collect import EpisodeConfig, run_episode, save_episode, pack_dataset
print("repo", REPO, "| out", OUT)

## 1. The room

`RoomConfig(seed=0)` reproduces the Colab arena (12 x 12 m, 3 m walls, 14 pillars, same random draws).
To make the room bigger, change `size_x`, `size_y`, `num_pillars`; to add interior walls or doorways, pass `extra_boxes`.
Numbers on the pillars are their heights: pillars lower than the flight band are flown over.

In [ ]:
ROOM = RoomConfig(
    size_x=12.0, size_y=12.0,      # meters
    num_pillars=14,
    seed=0,
    # Example of an extra interior wall with a 2 m doorway (uncomment to try):
    # extra_boxes=[Box(center=(0.0, 3.5, 1.5), size=(0.2, 5.0, 3.0), color=(0.7, 0.7, 0.7), kind="extra")],
)
room = Room(ROOM)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, seed in zip(axes, [0, 1, 2]):
    Room(RoomConfig(**{**ROOM.to_dict(), "seed": seed, "extra_boxes": ROOM.extra_boxes})).plot(ax=ax)
plt.tight_layout(); plt.show()

## 2. Start, goal and a collision-free path

Set `START` and `GOAL` to `[x, y, z]` in meters to choose them yourself, or leave them `None` for random ones
(at least `min_distance` apart, heights from `z_range`). `clearance` is how far the path stays from obstacles
(drone half-size about 0.15 m plus margin).

In [ ]:
PLANNER = PlannerConfig(resolution=0.1, clearance=0.45, z_range=(1.0, 2.0), min_distance=3.0)
START = None          # e.g. [-4.0, -4.0, 1.2]
GOAL = None           # e.g. [4.0, 4.5, 1.8]

rng = np.random.default_rng(0)
if START is None or GOAL is None:
    START, GOAL, path = sample_start_goal(room, PLANNER, rng)
else:
    START, GOAL = np.array(START, float), np.array(GOAL, float)
    path = plan_path(room, START, GOAL, PLANNER)
    assert path is not None, "no collision-free path between these points"
length = np.linalg.norm(np.diff(path, axis=0), axis=1).sum()
print("start", START.round(2), "goal", GOAL.round(2), f"| path {len(path)} points, {length:.1f} m")
room.plot(path=path, start=START, goal=GOAL); plt.show()

## 3. Build the Genesis scene (GPU from here on)

Building compiles Genesis kernels, which takes a while the first time. Rendering one 224 x 224 frame should take
milliseconds on the GPU; if it takes around a second, Genesis fell back to CPU rendering (check the `[egl]` line in cell 0).

In [ ]:
import torch
from drone_sim.env import DroneFPVEnv
t = time.time()
env = DroneFPVEnv(ROOM, image_size=224, keep_frames=True)
print(f"build {time.time() - t:.0f} s | device {env.device} | GPU {torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}")
env.render(); t = time.time(); rgb, depth = env.render(); print(f"render {1000 * (time.time() - t):.1f} ms/frame")
obs = env.reset(position=START)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(obs["rgb"]); ax[0].set_title("RGB at start")
im = ax[1].imshow(obs["depth"], cmap="viridis", vmin=0, vmax=12); ax[1].set_title("depth (m)"); plt.colorbar(im, ax=ax[1])
for a in ax: a.axis("off")
plt.show()

## 4. Fly the planned path once and look at it

The follower turns the drone to face its direction of travel (so the forward camera sees where it goes),
flies at `cruise_speed`, slows near the goal and hovers `hover_steps` steps at the end.
`action_noise_std` adds small held random perturbations so the dataset is not only perfectly smooth actions.

In [ ]:
FOLLOWER = FollowerConfig(cruise_speed=1.0, lookahead=0.6, goal_tolerance=0.25, action_noise_std=0.0,
                          yaw_rate_scale=env.yaw_rate_scale)
EPISODE = EpisodeConfig(hover_steps=5, stop_on_collision=True, start_yaw="path")

ep = run_episode(env, PLANNER, FOLLOWER, EPISODE, np.random.default_rng(1), start=START, goal=GOAL)
m = ep["meta"]
print(f"{m['outcome']} | {m['steps']} steps ({m['steps'] * 0.1:.1f} s sim) in {m['seconds']:.1f} s wall, "
      f"{m['steps_per_second']:.1f} steps/s | goal error {m['final_goal_distance_m']:.2f} m | "
      f"min clearance {m['min_clearance_m']:.2f} m")
print({k: (v.shape, v.dtype) for k, v in ep.items() if k != "meta"})

fig = plt.figure(figsize=(16, 6))
ax = fig.add_subplot(1, 3, 1); room.plot(ax=ax, path=ep["path"], flown=ep["state"][:, :3], start=START, goal=GOAL)
ax = fig.add_subplot(1, 3, 2); t = np.arange(len(ep["state"])) * 0.1
for i, n in enumerate("xyz"): ax.plot(t, ep["state"][:, i], label=n)
ax.set_xlabel("time (s)"); ax.set_ylabel("position (m)"); ax.legend(); ax.grid(alpha=0.3)
ax = fig.add_subplot(1, 3, 3)
for i, n in enumerate(["forward", "left", "up", "yaw_rate"]): ax.plot(t[:-1], ep["action"][:, i], label=n)
ax.set_xlabel("time (s)"); ax.set_ylabel("action"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

picks = np.linspace(0, len(ep["rgb"]) - 1, 6).astype(int)
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for c, i in enumerate(picks):
    axes[0, c].imshow(ep["rgb"][i]); axes[0, c].set_title(f"t = {i / 10:.1f} s")
    axes[1, c].imshow(ep["depth"][i].astype(np.float32), cmap="viridis", vmin=0, vmax=12)
    axes[0, c].axis("off"); axes[1, c].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Video of the flight (RGB | depth side by side)
import imageio.v2 as imageio
from IPython.display import Video
d = np.clip(ep["depth"].astype(np.float32) / 12.0, 0, 1)
d = (plt.cm.viridis(d)[..., :3] * 255).astype(np.uint8)
os.makedirs(OUT, exist_ok=True)
video = f"{OUT}/flight_preview.mp4"
imageio.mimsave(video, list(np.concatenate([ep["rgb"], d], axis=2)), fps=10)
Video(video, embed=True, width=640)

## 5. Collect the first few trajectories

Random start/goal pairs in this room, a random cruise speed per episode and small action noise.
Each episode is saved right away to `OUT/raw/room<seed>_ep<k>.npz`; rerunning skips existing files.
Same settings as `scripts/drone/collect.py`, so these episodes can be mixed with the batch ones.

In [ ]:
N_EPISODES = 5
FOLLOWER_COLLECT = FollowerConfig(action_noise_std=0.1, yaw_rate_scale=env.yaw_rate_scale)
EPISODE_COLLECT = EpisodeConfig(speed_range=(0.6, 1.5))
env.keep_frames = False
os.makedirs(f"{OUT}/raw", exist_ok=True); os.makedirs(f"{OUT}/logs", exist_ok=True)
for k in range(N_EPISODES):
    path_k = f"{OUT}/raw/room{ROOM.seed:03d}_ep{k:05d}.npz"
    if os.path.exists(path_k):
        continue
    ep_k = run_episode(env, PLANNER, FOLLOWER_COLLECT, EPISODE_COLLECT, np.random.default_rng([1000 + ROOM.seed, k]))
    save_episode(ep_k, path_k)
    mk = ep_k["meta"]
    with open(f"{OUT}/logs/notebook.jsonl", "a") as f:
        f.write(json.dumps({"file": os.path.basename(path_k), **{x: mk[x] for x in ("outcome", "steps", "path_length_m", "final_goal_distance_m", "min_clearance_m", "cruise_speed", "seconds")}}) + "\n")
    print(f"ep {k}: {mk['outcome']:9s} {mk['steps']:4d} steps, path {mk['path_length_m']:.1f} m, "
          f"goal err {mk['final_goal_distance_m']:.2f} m, {mk['steps_per_second']:.1f} steps/s")

## 6. Pack into the DINO-WM layout and load it with the training loader

Same layout as the PointMaze dataset (`states.pth`, `actions.pth`, `seq_lengths.pth`, `obses/episode_XXX.pth`)
plus `depths/episode_XXX.pth` (meters) and `episodes.json`. The loader is `datasets/drone_dset.py`
(`conf/env/drone_nav.yaml` for training).

In [ ]:
info = pack_dataset(f"{OUT}/raw/*.npz", DSET)
print(json.dumps({k: info[k] for k in ("n_episodes", "max_len", "total_frames", "outcomes", "rooms")}, indent=1))

from datasets.drone_dset import load_drone_slice_train_val
from datasets.img_transforms import default_transform
slices, trajs = load_drone_slice_train_val(default_transform(img_size=224), data_path=DSET, split_ratio=0.8,
                                           num_hist=3, num_pred=1, frameskip=5, return_depth=True, depth_size=224)
obs, act, state = slices["train"][0]
print("visual", tuple(obs["visual"].shape), "| depth", tuple(obs["depth"].shape), "| proprio", tuple(obs["proprio"].shape),
      "| action (frameskip 5 concatenated)", tuple(act.shape), "| state", tuple(state.shape))

## Next

- Happy with the trajectories? Shut this session down and run the batch job:
  `EPISODES=200 sbatch --array=0-9 slurm/drone_collect.sbatch` (10 rooms, room seed = array index), then pack with
  `python scripts/drone/collect.py --pack --out $WM/data/drone_raw --dset $WM/data/drone_nav`.
- First DINO-WM sanity check: train on one room only (`--array=0-0`), then compare with several rooms and `val_rooms`.